# sgd-vanilla-from-scratch — worked example 3: SGD converges on a shifted quadratic loss

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `sgd-vanilla-from-scratch`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

For a quadratic loss L(w) = 0.5*(w - w*)^2, the gradient is exactly w - w*, making this a clean analytical test bed. Starting from any w0, SGD with a small enough learning rate will converge to w* with exponentially decaying residuals. Each step multiplies the error (w - w*) by (1 - lr).

## Worked solution

**Step 1 — Initialize a MiniTensor at w0=5.** The true minimum is at w*=1.

**Step 2 — Run 30 SGD steps.** At each step: compute loss, set grad = w - w*, call sgd_step, record loss.

**Step 3 — Check loss is decreasing.** Each loss should be strictly less than the previous (for lr=0.1, (1-lr)^2 = 0.81 per step).

**Step 4 — Check convergence.** After 30 steps with lr=0.1, `(w - w*)^2 * 0.5` should be very small: `0.5 * (5-1)^2 * (0.9)^60 ≈ 0.006`.

In [ ]:
import torch as t

t.manual_seed(0)

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = array
        self.requires_grad = requires_grad
        self.grad = None
        self.recipe = None

def sgd_step(params, lr):
    for p in params:
        if p.grad is None:
            continue
        p.array -= lr * p.grad
        p.grad = None

w0, w_star, lr, n_steps = 5.0, 1.0, 0.1, 30
w = MiniTensor(t.tensor([w0]), requires_grad=True)

losses = []
for step in range(n_steps):
    loss = 0.5 * (w.array.item() - w_star) ** 2
    losses.append(loss)
    w.grad = w.array - w_star
    sgd_step([w], lr)

print(f'Initial loss: {losses[0]:.4f}')
print(f'Final loss after {n_steps} steps: {losses[-1]:.6f}')
print(f'Monotone decreasing: {all(losses[i] > losses[i+1] for i in range(len(losses)-1))}')
print(f'Final w: {w.array.item():.4f}  (should be near {w_star})')